<a href="https://colab.research.google.com/github/lasigeBioTM/data-text-processing-notebooks/blob/main/notebooks/03-data-extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unix Shell Tutorial: Filtering and Extracting Biomedical Data


This is the **third tutorial** in a series that will demonstrate how shell scripting can be used to perform the tasks that health and life science specialists may need to undertake to find and retrieve biomedical data and text. We will use the compound caffeine as an example and explore different public repositories to identify diseases related to it. The focus is not on the specific relationships we may discover, but on the process of obtaining them.

The objective of this tutorial is to learn how to efficiently filter and extract relevant data from the CSV file retrieved in the previous tutorial. Specifically, we will focus on filtering for proteins associated with putative caffeine-related diseases and extracting only the corresponding protein identifiers.

> This tutorial is part of a series of tutorials adapted as interactive versions of the hands-on steps described in the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book, which is licensed under the [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

## Step 1: Filtering Relevant Data with `grep`


Some data in the CSV file may not be relevant for our information need, so we may need to identify and extract only the relevant rows and columns. In this tutorial, we will first select relevant proteins (rows) using the command-line tool `grep`, and then select specific columns using the command-line tool `cut`.

Because our running example is caffeine, we will use a simple heuristic and keep only proteins whose identifiers include the species suffixes `_HUMAN`, `_RAT`, or `_MOUSE`.

Extracting lines from a text file is the main function of `grep`. Selection is performed by providing a pattern that `grep` searches for in each line and returning only matching lines. `grep` also supports more complex patterns such as regular expressions, which we will use later.

To get started, we first need to retrieve the data file generated in the previous tutorial. The following command downloads the `chebi_27732_xrefs_UniProt.csv` file directly from the GitHub repository:


In [36]:
%%bash
curl -s -O 'https://raw.githubusercontent.com/lasigeBioTM/data-text-processing-notebooks/refs/heads/main/data/chebi_27732_xrefs_UniProt.csv'

### Single and Multiple Patterns


We can execute the following command to select proteins that contain `_RAT` as the species suffix in the UniProt entry identifier (for example, `RYR1_RAT`):


In [37]:
%%bash
grep '_RAT' chebi_27732_xrefs_UniProt.csv


"27732","CHEBI:27732","chebi","RYR1_RAT","F1LMY4","uniprot"


**Expected Output:** A shorter list of proteins, all containing `_RAT` as the species suffix in the UniProt entry identifier.


Note that, if instead we select proteins that contain `RAT` **without the underscore**, we may get unwanted matches if `_ARATH` (a small flowering plant, *Arabidopsis thaliana*) was part of the list, because `RAT` appears in both suffixes.

**Expected Output:** A list of proteins not only for `_RAT` but also for


## Step 2: Multiple Pattern Matching


To use multiple patterns, we can repeat the `-e` option once per pattern:


In [38]:
%%bash
grep -e '_HUMAN' -e '_RAT' -e '_MOUSE' chebi_27732_xrefs_UniProt.csv


"27732","CHEBI:27732","chebi","RYR1_MOUSE","E9PZQ0","uniprot"
"27732","CHEBI:27732","chebi","RYR1_HUMAN","P21817","uniprot"
"27732","CHEBI:27732","chebi","RYR1_RAT","F1LMY4","uniprot"


**Expected Output:** A longer list of proteins matching any of the three species suffixes (`_HUMAN`, `_RAT`, or `_MOUSE`).

Tip: you can repeat `-e PATTERN` multiple times; `grep` will match a line if any of the patterns match.


In a terminal, we would typically use `| less` to scroll through long outputs. However, in a notebook environment, using `less` does not work well because it requires interactive input. Instead, we can use the `head` command to preview just the first few lines of the output. The `-n 10` option tells `head` to show only the first 10 lines.

In [39]:
%%bash
grep -e '_HUMAN' -e '_RAT' -e '_MOUSE' chebi_27732_xrefs_UniProt.csv | head -n 10


"27732","CHEBI:27732","chebi","RYR1_MOUSE","E9PZQ0","uniprot"
"27732","CHEBI:27732","chebi","RYR1_HUMAN","P21817","uniprot"
"27732","CHEBI:27732","chebi","RYR1_RAT","F1LMY4","uniprot"


**Expected Output:** The first 10 lines of the filtered protein list.

## Creating and Updating the Script

We can now update our script file to contain the following lines:

```bash
url="https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/$1/xref/UniProtKB?size=99&format=csv"

curl -s "$url" | \
  grep -e '_HUMAN' -e '_RAT' -e '_MOUSE'
```

We added the `-s` option to suppress the progress information printed by `curl`. The trailing `\` characters indicate line continuation; they must be the last characters on the line (no trailing spaces).


In [40]:
%%bash
cat > getproteins.sh << 'EOF'
url="https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/$1/xref/UniProtKB?size=99&format=csv"

curl -s "$url" | \
  grep -e '_HUMAN' -e '_RAT' -e '_MOUSE'
EOF


**Expected Output:** Script file `getproteins.sh` created successfully.

In [41]:
%%bash
chmod u+x getproteins.sh

**Expected Output:** No output (permissions set successfully).

In [42]:
%%bash
./getproteins.sh 27732

"27732","CHEBI:27732","chebi","RYR1_MOUSE","E9PZQ0","uniprot"
"27732","CHEBI:27732","chebi","RYR1_HUMAN","P21817","uniprot"
"27732","CHEBI:27732","chebi","RYR1_RAT","F1LMY4","uniprot"


**Expected Output:** Filtered list of relevant proteins for caffeine (CHEBI:27732).

In [43]:
%%bash
./getproteins.sh 27732 > chebi_27732_xrefs_UniProt_relevant.csv

**Expected Output:** No output (file saved successfully).

## Step 3: Data Elements Selection with `cut`


Now we need to select specific columns from the CSV file. Selecting columns from a delimited text file is a common task for `cut`. The `cut` command can receive the delimiter character with `-d` and the fields (columns) to extract with `-f`.

In our CSV file, the **fifth** column contains the UniProt accession (for example, `P21817`). As a quick demonstration, the first column is the numeric ChEBI identifier (`27732`).


In [44]:
%%bash
cut -d, -f1 < chebi_27732_xrefs_UniProt_relevant.csv

"27732"
"27732"
"27732"


**Expected Output:** Only the first column of the file (the numeric ChEBI identifier).

In CSV files, commas (`,`) separate the columns. The command below prints only the first column.


We can also extract multiple columns at once by separating the column numbers with a comma. For example, to get both the first column (the ChEBI identifier) and the fifth column (the UniProt identifier), we use `-f1,5`:

In [45]:
%%bash
cut -d, -f1,5 < chebi_27732_xrefs_UniProt_relevant.csv

"27732","E9PZQ0"
"27732","P21817"
"27732","F1LMY4"


**Expected Output:** First and fifth columns of the file.

Now, the output contains both the first and fifth column of the file.

## Final Script with Column Selection

We can now update our script file to output **only** UniProt accessions (column 5). The script will:
1) retrieve the cross-references for a ChEBI identifier,
2) keep only `_HUMAN`, `_RAT`, and `_MOUSE` entries,
3) extract the UniProt accession column, and
4) remove quotation marks.

```bash
url="https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/$1/xref/UniProtKB?size=99&format=csv"

curl -s "$url" | \
  grep -e '_HUMAN' -e '_RAT' -e '_MOUSE' | \
  cut -d, -f5 | \
  tr -d '"'
```

The last two commands extract field 5 (the UniProt accession column) and remove the double-quote characters that appear in the CSV output.


In [46]:
%%bash
cat > getproteins.sh << 'EOF'
url="https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/$1/xref/UniProtKB?size=99&format=csv"

curl -s "$url" | \
  grep -e '_HUMAN' -e '_RAT' -e '_MOUSE' | \
  cut -d, -f5 | \
  tr -d '"'
EOF


**Expected Output:** Final script version created.

Now we can execute the script for caffeine:

In [47]:
%%bash
./getproteins.sh 27732

E9PZQ0
P21817
F1LMY4


**Expected Output:** Only protein identifiers (fifth column, with quotes removed) for proteins from HUMAN, RAT, and MOUSE species.

In [48]:
%%bash
./getproteins.sh 27732 > chebi_27732_xrefs_UniProt_relevant_identifiers.csv

**Expected Output:** No output (file saved successfully).

To check if the file was really created and to analyze its contents, we can
use the `cat` command:

In [49]:
%%bash
cat chebi_27732_xrefs_UniProt_relevant_identifiers.csv

E9PZQ0
P21817
F1LMY4


**Expected Output:** List of protein identifiers associated with caffeine-related diseases.

# Conclusion

This concludes the **Filtering and Extraction** tutorial adapted from the [Data and Text Processing for Health and Life Sciences](https://labs.rd.ciencias.ulisboa.pt/book/) book.

In this tutorial, we practiced filtering and extracting structured data using `grep`, `head`, `cut`, and `tr`, and we combined them into a small reusable shell script.

The next tutorial in this series will explore task repetition techniques to efficiently apply the same task to all proteins in the list we have gathered.


# Exercise

As an exercise, execute the script to extract only the protein identifiers associated with [Melatonin](https://www.ebi.ac.uk/chebi/searchId.do?chebiId=CHEBI:16796).


In [60]:
%%bash
./getproteins.sh 16796

**Expected Output:** No output is given since Malatonin has hundreds of proteins and only the last ones are from common organisms.
Check the referenceCount value below:

In [64]:
%%bash
curl -s "https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/16796/xref/UniProtKB?size=1&format=xml"

<?xml version="1.0" encoding="UTF-8" standalone="yes"?><result><entries><entry acc="CHEBI:16796" id="16796" source="chebi"><referenceCount>366</referenceCount><references><reference acc="J3M8V6" id="J3M8V6_ORYBR" source="uniprot"/></references></entry></entries></result>


But if we start from protein 250, by adding `&start=250` to the url, we retrieve a more friendly list:



In [68]:
%%bash
curl -s "https://www.ebi.ac.uk/ebisearch/ws/rest/chebi/entry/16796/xref/UniProtKB?size=100&format=csv&start=250"

"org_id","org_acc","org_source","ref_id","ref_acc","ref_source"
"16796","CHEBI:16796","chebi","A0A453ALG6_AEGTS","A0A453ALG6","uniprot"
"16796","CHEBI:16796","chebi","A0A452ZJV0_AEGTS","A0A452ZJV0","uniprot"
"16796","CHEBI:16796","chebi","A0A6A2ZGR9_HIBSY","A0A6A2ZGR9","uniprot"
"16796","CHEBI:16796","chebi","I1QS93_ORYGL","I1QS93","uniprot"
"16796","CHEBI:16796","chebi","A0AAV5LPA8_9ROSI","A0AAV5LPA8","uniprot"
"16796","CHEBI:16796","chebi","A0AAE0CNP2_9ROSI","A0AAE0CNP2","uniprot"
"16796","CHEBI:16796","chebi","A0A0D3G9R8_9ORYZ","A0A0D3G9R8","uniprot"
"16796","CHEBI:16796","chebi","A0ACC3LVF6_EUCGR","A0ACC3LVF6","uniprot"
"16796","CHEBI:16796","chebi","A0A453M999_AEGTS","A0A453M999","uniprot"
"16796","CHEBI:16796","chebi","A0A452ZJS7_AEGTS","A0A452ZJS7","uniprot"
"16796","CHEBI:16796","chebi","A0A0D3HU33_9ORYZ","A0A0D3HU33","uniprot"
"16796","CHEBI:16796","chebi","A0A8S1ZNP7_ARAAE","A0A8S1ZNP7","uniprot"
"16796","CHEBI:16796","chebi","A0A9R0V039_TRITD","A0A9R0V039","uniprot"
"16796",

**Expected Output:** File `gold_proteins.csv` created with protein identifiers for gold (CHEBI:30050).

In [52]:
%%bash
cat adenosine_proteins.csv

**Expected Output:** List of protein identifiers associated with water.

In [53]:
%%bash
cat gold_proteins.csv

**Expected Output:** List of protein identifiers associated with gold.